In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
from scipy import optimize
import matplotlib.pyplot as plt


# plotting
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# import py-files and PACKAGE HERE
from copy import deepcopy
from SolowModel import SolowModelClass

# 2. The Solow Model with a Time-varying Savings Rate
## Question 2.1
### Question 2.1.1
We start by finding the steady state values for capital, output and consumption per worker using both steady_state and solve_steady_state, that utilises a root optimizer

In [ ]:
baseline= SolowModelClass()
print(baseline)

#1. we call the two defined functions and compare them
k_ss, y_ss, c_ss = baseline.steady_state()
k_ss_root = baseline.solve_steady_state()
print(f'\nk* = {k_ss:.4f}, y* = {y_ss:.4f}, c* = {c_ss:.4f}')
print(f'k* (root-finder) = {k_ss_root:.4f}')

print(f'It is {np.isclose(k_ss, k_ss_root)} that k* and k* with rootfinder is identical')

So we find that finding steady state for capital numerically or analytically yield the same results.

### Question 2.1.2
Now, we simulate the model and plot the development in the savings rate, capital and consumption. We also report $c_0, c_{T-1}$ and $k_{T-1}$:

In [ ]:
sim=baseline.simulate(baseline.par.s_bar, k0=0.1)


def plot_series(ax, t, sims, key, ylabel): # define the plots as function, as it will be useful later
    """ plot one variable (key: 's', 'k', or 'c') for every simulation in sims """

    for label, sim in sims.items():
        style = dict(ls='--', color='black', lw=2, zorder=10) if label == 'baseline' else {}
        ax.plot(t, getattr(sim, key), label=label, **style)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10)

    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10)

def plot_solow_paths(t, sims, title):

    fig = plt.figure(figsize=(13,4))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    ax1 = fig.add_subplot(1,3,1)
    plot_series(ax1, t, sims, 's', '$s_t$')

    ax2 = fig.add_subplot(1,3,2)
    plot_series(ax2, t, sims, 'k', '$k_t$')
    ax2.set_xlabel('$t$')

    ax3 = fig.add_subplot(1,3,3)
    plot_series(ax3, t, sims, 'c', '$c_t$')
    
    fig.tight_layout()

    return fig

t = np.arange(baseline.par.T) #construct the x-axis
sims = {'baseline': sim}
fig = plot_solow_paths(t, sims, 'Baseline simulation of savings, capital and consumption')

#and now we report the chosen values
print(f'c0 = {sim.c[0]:.4f}, c_T-1 = {sim.c[-1]:.4f}, k_T-1 = {sim.k[-1]:.4f}')

So we get that in the last period, consumption and capital matches the steady state values. From the figures, we see that both values converge to steady state after around 10 periods, where as the savings rate is constant (like defined).

### Question 2.1.3
We now test the model and how it is made with two assert statements. We will here utilise that we know that an economy starting in steady state should stay there - we therefore insert k_ss into the simulate function. Next, we know that if the savings rate is 0, there is a closed form equation for Capital. We calculate the close form and compare it with what the simulation function finds:

In [ ]:
#capital stays in steady state:
sim_ss=baseline.simulate(baseline.par.s_bar, k0=k_ss)

assert np.allclose(sim_ss.k, k_ss), \
    f'k should stay at {k_ss:.4f}, but ranges from {sim_ss.k.min():.4f} to {sim_ss.k.max():.4f}'

print(f'The path of k stays between {sim_ss.k.min():.4f} and {sim_ss.k.max():.4f}')

#capital decays according to the closed form when s=0:
sim_k = baseline.simulate(s=0, k0=0.1)
closed_form = (1-baseline.par.delta)**np.arange(baseline.par.T)*baseline.par.k0
max_diff = np.max(np.abs(sim_k.k - closed_form))

assert np.allclose(sim_k.k, closed_form), \
    f'max deviation from closed form is {max_diff:.2e}, should be close to 0'

print(f'k decays from {sim_k.k[0]:.4f} to {sim_k.k[-1]:.4f}, matching the closed form to within {max_diff:.2e}')

## Question 2.2
### Question 2.2.1
We implement the savings rule in equation 6 into the model and call it s_path. It gives us the following plot, showing that the savings rate converges to $\bar s$ over time

In [ ]:
t = np.arange(baseline.par.T)

plt.plot(t, baseline.s_path(0.35, 0.75), label=r'$s_0=0.35, \varphi=0.75$')
plt.axhline(baseline.par.s_bar, ls='--', color='black', label=r'$\bar{s}$')
plt.xlabel('t')
plt.ylabel('$s_t$')
plt.legend(fontsize=10)
plt.show()

### Question 2.2.2
We now simulate the model with the different values of the initial savings rate and the speed to which it returns to s_bar. From there, we feed that into the simulate and plot $s_t$, $k_t$ and $c_t$ and compare it to the baseline

In [ ]:
s0phi = {r'$s_0=0.30, \varphi=0.50$': (0.30,0.50), r'$s_0=0.40, \varphi=0.80$': (0.40,0.80),
         r'$s_0=0.10, \varphi=0.50$': (0.10,0.50), r'$s_0=0.60, \varphi=0.60$': (0.60,0.60)} #the different simulations

#change labels
sim_baseline = deepcopy(baseline.simulate(baseline.par.s_bar))
sims = {'baseline': deepcopy(sim_baseline)}

for label, (s0, phi) in s0phi.items():
    sr = baseline.s_path(s0, phi)
    sim_i = baseline.simulate(sr)
    sims[label] = deepcopy(sim_i) #hold the simulations

#Now plot it up against the baseline
fig = plot_solow_paths(t, sims, 'Savings rules vs. baseline')

So we find that the path towards steady state differs based on the initial savings rate and speed of convergence to baseline savings rate, but it looks like all reach the steady state.

### Question 2.2.3
We report the capital per worker in the last period and see if they are the same:

In [ ]:
for label, sim in sims.items():
    print(f'{label:20s}: k_T-1 = {sim.k[-1]:.4f}')

So as expected, regardless $s_0$ and $\varphi$, all 4 simulations converge to the same steady state value of $k$ by the final period, matchin the value found in question 1 of k=0.7607. This is because all rules share the same long-run savings rate, s=0.25, and since $\varphi<1$ in every case, eventually all simulations will end in the same steady state and only the transition path differs.

### Question 2.2.4
$\varphi$ governs how persistently the savings rate stays away from $\bar s$, the long-run savings rate: the closer $\varphi$ is to 1, the more periods are $s_t$ different from $\bar s$. $s_0$ governs the initial size of that deviation. The two interact in shaping the transition path rather than acting separately: whether $k_t$ and $c_t$ temporarily are higher than their steady-state values depends on both. Overshoot is only possible when $s_0>\bar s$ — saving more than the long-run rate is what can push capital past where it eventually settles; when $s_0<\bar s$ (as in $s_0=0.10$), the economy simply approaches $k^*$ from below without ever exceeding it. But $s_0>\bar s$ alone isn't sufficient either: $(s_0=0.30,\varphi=0.50)$ also starts above $\bar s$ but does not overshoot, because the deviation decays too quickly to accumulate enough extra investment. It's the combination of the two values that determines whether the extra savings compounds into values higher than the steady state, as seen in $(s_0=0.40,\varphi=0.80)$ and $(s_0=0.60,\varphi=0.60)$.


## Question 2.3
### Question 2.3.1 and 2.3.2
We now incorporate welfare into the model in the python file.
we compute the welfare for the baseline and the 4 simulations:

In [ ]:
W_baseline = baseline.welfare(sims['baseline'].c)
rows = {label: baseline.welfare(sim.c) for label, sim in sims.items()}
W_table = pd.DataFrame({'W': rows})
W_table['W - W_baseline'] = W_table['W'] - W_baseline
W_table.round(4)

### Question 2.3.3
So we see that only one of the simulations beats the baseline/long-run savings rate: $s_0=0.30$ and $\varphi=0.50$ (the blue line in the plot). Here, they save more than the long-run savings rate in the beginning, but quickly converge to the long-run rate. The other simulations give lower welfare because of the tradeoff in the model between early consumption, which is highly valued because of the discount rate $\beta$, and more accumulated capital, which leads to higher consumption in later periods. When $s_0=0.10$, consumption is high in the beginning, but it results in less accumulated capital, meaning less consumption later. When $s_0=0.40$ and $s_0=0.60$, they save more in the beginning, meaning more consumption later, but it does not outweigh the initial sacrifice in consumption.

When looking at Question 2, we see that this simulation is the one that most closely follows the baseline — usually you would look for the biggest difference between the graphs, and therefore we do not think we would have found this result just by looking at Question 2 alone.

## Question 2.4
### Question 2.4.1
To evaluate many combinations of $s_0$ and $\varphi$ without repeating the same three lines everywhere, we implement the evaluate equation in the solow model, that returns the welfare of a given savings rule directly. We then make a grid of the possible $s_0$ and $\varphi$, and from there we plot a 3d plot and a contour plot. From there, we report the values that give the highest welfare:

In [ ]:
#the grid
N=100
s0_vec = np.linspace(0, 0.6, N) 
phi_vec = np.linspace(0, 0.95,N)

# and we find the welfare for each of them, running the savings rate and consumption functions before
W_grid = np.empty((N, N))
for i, s0 in enumerate(s0_vec):
    for j, phi in enumerate(phi_vec):
        W_grid[i,j] = baseline.evaluate(s0, phi)

#point that gives the highest welfare:
i_best, j_best = np.unravel_index(np.argmax(W_grid), W_grid.shape)
s0_best, phi_best, W_best = s0_vec[i_best], phi_vec[j_best], W_grid[i_best, j_best]
print(f's0={s0_best:.4f}, phi={phi_best:.4f}, W={W_best:.4f}')

#the plot
s0_grid, phi_grid = np.meshgrid(s0_vec, phi_vec,indexing='ij') # for the contour plot
fig = plt.figure(figsize=(13,5.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1], wspace=0.08) 

# a. The 3d plot:
ax1 = fig.add_subplot(gs[0], projection='3d')
surf = ax1.plot_surface(s0_grid, phi_grid, W_grid, cmap='viridis')


# b . labels and titles:
ax1.set_title('Welfare over the grid')
ax1.set_xlabel('$s_0$')
ax1.set_ylabel(r'$\varphi$')
ax1.set_zlabel('$W$')

# c. change the direction of the y-axis and where the z-axis is placed
ax1.view_init(azim=115,elev=30) 
ax1.set_box_aspect([4,4,3],zoom=1.15) 

# the contour plot
ax2 = fig.add_subplot(gs[1])
cs = ax2.contourf(s0_grid, phi_grid, W_grid, levels=30, cmap='viridis')
fig.colorbar(cs, ax=ax2, shrink=0.85)
ax2.plot(s0_best, phi_best, 'o', color='red', ms=10, label='best on grid')
ax2.set_title('Contour, with the best point marked')
ax2.set_xlabel('$s_0$')
ax2.set_ylabel(r'$\varphi$')
ax2.legend()

fig.subplots_adjust(left=0.02, right=0.98, top=0.85, bottom=0.14)

So we find that welfare can be improved if the $s_0=0.3091$ and $ \varphi=0.2303$, giving a welfare of -3.9843. 

### Question 2.4.2
We now run a numerical optimizer and compare. We utilise 'SLSQP' as our method because the objective is smooth and is good at handling bounds: 

In [ ]:

def objective(x):
    s0, phi = x
    return -baseline.evaluate(s0, phi)  # - in front because we want to maximize

gridpoint = [s0_best, phi_best] # the best point from the grid search
bounds = [(0, 0.60), (0, 0.95)]

result = optimize.minimize(objective, gridpoint, method='SLSQP', bounds=bounds)
print('success:', result.success)

s0_opt, phi_opt = result.x
W_opt = -result.fun

#compare the results
print(f'grid:      s0={s0_best:.4f}, phi={phi_best:.4f}, W={W_best:.4f}') 
print(f'optimizer: s0={s0_opt:.4f}, phi={phi_opt:.4f}, W={W_opt:.4f}')
print(f'improvement over grid: {W_opt - W_best:.6f}')

So we find that solving the maximisation problem through a grid search or a numerical optimizer yield very similar results for welfare, but they report different initial savings rates where the optimizer finds a slightly higher value. When looking at the contour plot, it seems there is a bigger surface yielding a very similar welfare, which could explain the difference.

### Question 2.4.3
We now plot the convergence path for savings, capital and consumption for the baseline and the optimizer. We also report the initial consumption and the first period consumption is over baseline.

In [ ]:
#run the simulation with the optimal points
sr_opt = baseline.s_path(s0_opt, phi_opt)
sim_opt = deepcopy(baseline.simulate(sr_opt))

sims = {'baseline': sim_baseline, 'optimal rule': sim_opt}

fig = plot_solow_paths(t, sims, 'Best savings rule vs. baseline')

print(f'c0 (optimal rule) = {sim_opt.c[0]:.4f}')

above = np.where(sim_opt.c > sim_baseline.c)[0]
if above.size > 0:
    print(f'first period where c_t exceeds baseline: t = {above[0]}')
else:
    print('c_t never exceeds baseline within the horizon')

So we find that the path of convergence to steady state looks very similar, apart from the savings rate in the beginning, however it quickly converges to $s_bar$. Already at t=1 the consumption exceed baseline, because with $s_0=0.32$ savings were higher in t=0, resulting in larger capital in the period after and therefore higher consumption.

## Question 2.5
### Question 2.5.1
We now also let the long-run savings rate differ from $\bar s=0.25$. We again use the optimizer, but now extend the objective to also include s_long_run and compare the welfare value to the value found in question 4.

In [ ]:
def objective(x):
    s0, phi, s_inf = x
    return -baseline.evaluate(s0, phi, s_long_run=s_inf)  # - in front because we want to maximize

x0_3p = [s0_opt, phi_opt, baseline.par.s_bar]  # the best points from before, and start with s_long_run=0.25
bounds_3p = [(0, 1), (0, 0.95),(0, 1)] #we chose that the bounds for s_0 and s_long run was from 0 to 1, as this is the norm for solow models

result_3p = optimize.minimize(objective, x0_3p, method='SLSQP', bounds=bounds_3p)
print('success:', result_3p.success)

s0_opt_3p, phi_opt_3p, s_long_opt_3p = result_3p.x
W_opt_3p = -result_3p.fun

#compare the results
print(f'optimizer with two parameters:   s0={s0_opt:.4f}, phi={phi_opt:.4f}, W={W_opt:.4f}')
print(f'optimizer with three parameters: s0={s0_opt_3p:.4f}, phi={phi_opt_3p:.4f}, s_long_run={s_long_opt_3p:.4f}, W={W_opt_3p:.4f}')
print(f'improvement with 1 more parameter: {W_opt_3p - W_opt:.6f}')

So when we maximize welfare over three parameters instead of two, we find that welfare does improve. The long-run savings rate is 0.199, so lower than the previous, whilst the initial savings rate and especially phi is higher, meaning a prologed period of higher savings in the beginning. This shows us that welfare improves when holding a higher savings rate in the earlier periods, meaning a quicker accumulation in capital and allowing for lower savings rate in later periods.

### Question 2.5.2
We now compare capital per worker in the last period to the previously found steady state value. Because the long run savings rate now is free, we would expect the steady state of capital per worker to have changed since savings rate determines it. 

In [ ]:
#run the simulation with the optimal 3 parameters - start by simulating the savings path
sr_3p = baseline.s_path(s0_opt_3p, phi_opt_3p, s_long_run=s_long_opt_3p)
sim_3p = baseline.simulate(sr_3p)

print(f'k_T-1 (three-parameter rule) = {sim_3p.k[-1]:.4f}')
print(f'k* (when s=s_bar) = {k_ss:.4f}')
print(f'difference: {sim_3p.k[-1] - k_ss:.4f}')
print(f's_long_run (optimized) = {s_long_opt_3p:.4f}, s_bar (Question 1) = {baseline.par.s_bar:.4f}')

So we find that the capital per worker is lower when we use the three parameters to maximise welfare compared to the baseline - so the two model calibrations do not reach the same steady state, as we expected. The values should not be the same, since the long run savings rate is noticeably lower, resulting in a lower level of capital.

## Question 2.6
### Question 2.6.1
We now introduce a new functional form to the savings rate. In equation 6, we have that savings adjust back to the long rate at a constant rate each period, wher the gap between $s_t-\bar s$ shrinks by the proportional factor $\varphi$, regardless of how much time has passed. We now introduce an alternative functional form to savings, where the speed of adjustment is not constant, often called power-law decay:
$$s_t = s_{\infty} + (s_0-s_{\infty})\cdot\frac{1}{(1+\lambda t)^{\gamma}}, \qquad \lambda>0,\ \gamma>0$$

Where $\lambda$ sets the pace of adjustment and $\gamma$ changes the curvature of the decay, so how much of the gap remains after a given amount of time has passed.
Here, the savings rate move quickly away from the initial savings rate, $s_0$, but the remaining gap then closes more slowly, as t becomes larger. This is motivated by habit forming, and that it may be easy to change behaviour in the beginning but not in the long run, shown by long-memory time series processes, where the influence of an initial shock fades more gradually compared to equation (6). This functional form will still converge to a steady state, because when $t\to\infty$, then $\frac{1}{(1+\lambda t)^{\gamma}}\to 0$ for any $\lambda,\gamma>0$

### Question 2.6.2
We implement the function with a new function into solow model, defined as evaluate_ext, before we then pick bounds and a starting point to run optimize.

In [ ]:
def objective(x):
    s0, lam, gam, s_ext = x
    return -baseline.evaluate_ext(s0, lam, gam, s_long_run=s_ext)  # - in front because we want to maximize

x0_ext = [s0_opt_3p, 1.0, 1.0, s_long_opt_3p]  # use the starting point for savings from above, random guess for lam and gam
bounds_ext=[(0, 1), (1e-3, 5), (1e-3, 5), (0, 1)]  # s0, lam, gam, s_long_run, bound at 5 because otherwise not that different from 

result_ext = optimize.minimize(objective, x0_ext, method='SLSQP', bounds=bounds_ext)
print('success:', result_ext.success)

s0_opt_ext, lam_opt_ext, gam_opt_ext, s_long_opt_ext = result_ext.x
W_opt_ext = -result_ext.fun

print(f'optimizer with new savings function:   s0={s0_opt_ext:.4f}, lam={lam_opt_ext:.4f}, gam={gam_opt_ext:.4f}, s_long_run={s_long_opt_ext:.4f}, W={W_opt_ext:.4f}')

So we find that with this new savings rule that the initial savings rate that creates the highest welfare is very similar to question 5, and that the long run savings rate is slightly lower. 
### Question 2.6.3
We now compare welfare in a small table, using pandas.

In [ ]:
#dictionary for the names of the rows
W_compare = {
    'Question 4 (geometric, 2 free params)': W_opt,
    'Question 5 (geometric, 3 free params)': W_opt_3p,
    'Question 6 (power law, 4 free params)': W_opt_ext,
}

W_table_26 = pd.DataFrame({'W': W_compare})
W_table_26['W - W (Q4)'] = W_table_26['W'] - W_opt
W_table_26['W - W (Q5)'] = W_table_26['W'] - W_opt_3p
W_table_26.round(4)

So we find that with the new functional form for savings, welfare is higher than in question 4 but does not exceed welfare in question 5 - even though the new functional form have 1 more free parameter, and therefore more flexibility, it does not improve welfare. This suggests that Question 5's geometric rule already comes close to spanning the welfare-relevant shapes available to $s_t$, leaving little room for a differently-shaped rule to improve on it.